In [ ]:
import pandas as pd
import sqlite3
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from tqdm import tqdm

from unga80.config import DATA_DIR, ROOT_DIR
from unga80.database import QUERIES

DB = DATA_DIR / "countries.db"
model: str = "artifish/llama3.2-uncensored"

In [ ]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    rows = cursor.execute("select * from countries;").fetchall()
    column_names = [x[0] for x in cursor.description]
    cursor.close()

df = pd.DataFrame(rows, columns=column_names)
df.head()

## Word

In [ ]:
# Create a template with variables
template = """
Summarize the following speech in a single word that represent the content of the speech. 
Reply only a single word. 
Ignore the speaker and focus on the content:

Speech:
{text}
"""
model = "artifish/llama3.2-uncensored:latest"
llm = ChatOllama(
    model=model,
    temperature=0,
    keep_alive=False,
)
prompt = PromptTemplate.from_template(template)

column = "word"
update_word = f"""UPDATE countries SET {column} = ? where country = ?"""
max_length = 30
pbar = tqdm(df.iterrows(), total=df.shape[0])
for i, r in pbar:
    pbar.set_description(r["country"])
    if r["word"]:
        continue

    formatted_prompt = prompt.format(text=r["full_speech"])
    result = llm.invoke(formatted_prompt)
    if len(result.content) < max_length:
        with sqlite3.connect(DB) as conn:
            cursor = conn.cursor()
            cursor.execute(update_word, (result.content, r["country"]))
            conn.commit()
            cursor.close()

## Summary

In [ ]:
template = """
Summarize the following speech. Ignore the speaker and focus on the content:

{text}
"""
llm = ChatOllama(
    model=model,
    temperature=0,
)

prompt = PromptTemplate.from_template(template)

column = "summary"
update_countries = f"""UPDATE countries SET {column} = ? where country = ?"""

pbar = tqdm(df.iterrows(), total=df.shape[0])
for i, r in pbar:
    pbar.set_description(r["country"])
    if r["summary"]:
        continue

    formatted_prompt = prompt.format(text=r["full_speech"])
    result = llm.invoke(formatted_prompt)
    # print(result.content)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(update_countries, (result.content, r["country"]))
        conn.commit()
        cursor.close()

## Countries mentioned

In [ ]:
# template = """
# Identify the countries mentioned in the follwowing speech and identify the sentiment as positive, negative, or neutral towards that country. Format: [Country, Sentiment]. Ignore the information about the speaker.

# Speech:
# '''{text}'''
# """
template = """Analyze the following text \n\n{text} \n\nand identify all the country names mentioned.

Please provide two separate lists, one for positive mentions and one for negative mentions, of countries mentioned in the text, 
with each country followed by a brief description (positive/negative tone). For example: 'France - A culturally rich and 
historic country.' or 'North Korea - A disputed and isolated nation.'

Please note that the tone may be subjective, and the AI's interpretation should be based on the context and
language used in the original text.
"""
llm = ChatOllama(
    model=model,
    temperature=0,
)
prompt = PromptTemplate.from_template(template)

column = "countries_mentioned"
update_countries = f"""UPDATE countries SET {column} = ? where country = ?"""

pbar = tqdm(df.iterrows(), total=df.shape[0])
for i, r in pbar:
    pbar.set_description(r["country"])
    if r[column]:
        continue

    formatted_prompt = prompt.format(text=r["full_speech"])
    result = llm.invoke(formatted_prompt)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(update_countries, (result.content, r["country"]))
        conn.commit()
        cursor.close()

## Risks

In [ ]:
template = """Summarize the risks and challenges discussed in the provided speech.
Identify the most critical points that require attention.
Use bullet points to present a concise list of the main risks/concerns

Speech:
'''{text}'''
"""
llm = ChatOllama(
    model=model,
    temperature=0,
)
prompt = PromptTemplate.from_template(template)

column = "risks"
update_countries = f"""UPDATE countries SET {column} = ? where country = ?"""

pbar = tqdm(df.iterrows(), total=df.shape[0])
for i, r in pbar:
    pbar.set_description(r["country"])
    if r[column]:
        continue

    formatted_prompt = prompt.format(text=r["full_speech"])
    result = llm.invoke(formatted_prompt)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(update_countries, (result.content, r["country"]))
        conn.commit()
        cursor.close()

## Haiku

In [ ]:
template = """
Craft a 3-line haiku inspired by the key themes or messages of the speech. Reply only with the lines of the haiku.

Speech:
'''{text}'''
"""
llm = ChatOllama(
    model=model,
    temperature=0,
)
prompt = PromptTemplate.from_template(template)

column = "haiku"
update_countries = f"""UPDATE countries SET {column} = ? where country = ?"""

pbar = tqdm(df.iterrows(), total=df.shape[0])
for i, r in pbar:
    pbar.set_description(r["country"])
    if r[column]:
        continue

    formatted_prompt = prompt.format(text=r["full_speech"])
    result = llm.invoke(formatted_prompt)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(update_countries, (result.content, r["country"]))
        conn.commit()
        cursor.close()

# Yoda

In [ ]:
template = """
Craft a short advice in Yoda's voice, based on the content of the speech. Don't describe it, just write what Yoda would say.

Speech:
'''{text}'''
"""
llm = ChatOllama(model=model, temperature=0, keep_alive=False)
prompt = PromptTemplate.from_template(template)

output_yoda_path = ROOT_DIR / "output" / "yoda" / "text"
output_yoda_path.mkdir(parents=True, exist_ok=True)

pbar = tqdm(df.iterrows(), total=df.shape[0])
for i, r in pbar:
    pbar.set_description(r["country"])
    if (output_yoda_path / f"{r['country']}.txt").exists():
        continue

    formatted_prompt = prompt.format(text=r["full_speech"])
    result = llm.invoke(formatted_prompt)

    with open(output_yoda_path / f"{r['country']}.txt", "w") as f:
        f.write(str(result.content))

# George Carlin

In [ ]:
template = """
Craft a joke or a routine in George Carlin style based only on the content of the following speech. Don't describe it, just write what the comedian George Carlin would say.

Speech:
'''{text}'''
"""
llm = ChatOllama(model=model, temperature=0, keep_alive=False)
prompt = PromptTemplate.from_template(template)

output_carlin_path = ROOT_DIR / "output" / "george_carlin" / "text"
output_carlin_path.mkdir(parents=True, exist_ok=True)

pbar = tqdm(df.iterrows(), total=df.shape[0])
for i, r in pbar:
    pbar.set_description(r["country"])
    if (output_carlin_path / f"{r['country']}.txt").exists():
        continue

    formatted_prompt = prompt.format(text=r["full_speech"])
    result = llm.invoke(formatted_prompt)

    with open(output_carlin_path / f"{r['country']}.txt", "w") as f:
        f.write(str(result.content))

In [ ]:
# TODO what are the main messages from the speech
# TODO Identify the countries mentioned in the speech and identify the sentiment as positive, negative, or neutral towards that country. Format: [Country, Sentiment].
# TODO Summarize the key threats or challenges discussed in the speech by listing the top 3 risks or concerns in bullet points.
# TODO Craft a 3-line haiku inspired by the key themes or messages of the speech.
# TODO Craft a short advice in Yoda's voice, based on the content of the speech.
# TODO describe the speech in one word
# TODO a George Carlin joke from the speech